In [ ]:
! pip show langchain

In [2]:
! python --version

133.86s - pydevd: Sending message related to process being replaced timed-out after 5 seconds


Python 3.13.3


In [17]:
import os

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()  

llm = ChatOpenAI(
    model="mimo-v2.5-pro", 
    temperature=0.7,
    timeout=30,
    max_tokens=200,
    stop="我",
    api_key=os.environ["XIAOMI_API_KEY"],
    base_url="https://token-plan-cn.xiaomimimo.com/v1",
)

llm.invoke("介绍一下你自己")



AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 5, 'prompt_tokens': 253, 'total_tokens': 258, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 4, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 192}}, 'model_provider': 'openai', 'model_name': 'mimo-v2.5-pro', 'system_fingerprint': None, 'id': '21e50a11fcac49fc9e11bcb02b7eb1cc', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019e4324-4735-7b02-ade7-74947462ce7f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 253, 'output_tokens': 5, 'total_tokens': 258, 'input_token_details': {'cache_read': 192}, 'output_token_details': {'reasoning': 4}})

In [ ]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()  

llm = ChatOpenAI(
    model="mimo-v2.5-pro", 
    api_key=os.environ["XIAOMI_API_KEY"],
    base_url="https://token-plan-cn.xiaomimimo.com/v1",
)

question = "langchain 是什么？"
# llm.invoke(question)

# for chunk in llm.stream(question):
#     print(chunk.content + "|")

# llm.batch([question, "langchain 作者是谁"])


async for event in llm.astream_events(question, version="v2"):
  print(f"event: {event} | name: {event['name']} | data: {event['data']}")


In [26]:
import os
from typing import Optional
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field

load_dotenv()

llm = ChatOpenAI(
    model="mimo-v2.5-pro",
    api_key=os.environ["XIAOMI_API_KEY"],
    base_url="https://token-plan-cn.xiaomimimo.com/v1",
)


class Joke(BaseModel):
    """Joke to tell user"""
    setup: str = Field(description="The setup of the joke")
    punchline: str = Field(description="The punchline of the joke")
    rating: Optional[int] = Field(default=None, description="How funny the joke is, from 1 to 10")


# MiMo 默认 structured output 会混入 reasoning 文本，导致 JSON 解析失败
structured_llm = llm.with_structured_output(Joke, method="function_calling")
structured_llm.invoke("给我讲一个程序员的笑话")

Joke(setup='为什么程序员总是把万圣节和圣诞节搞混？', punchline='因为 Oct 31 = Dec 25 ！（八进制的31等于十进制的25）', rating=7)

In [27]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

llm = ChatOpenAI(
    model="mimo-v2.5-pro",
    api_key=os.environ["XIAOMI_API_KEY"],
    base_url="https://token-plan-cn.xiaomimimo.com/v1",
)

res = llm.invoke("介绍一下你自己")
res.usage_metadata

{'input_tokens': 253,
 'output_tokens': 283,
 'total_tokens': 536,
 'input_token_details': {'cache_read': 192},
 'output_token_details': {'reasoning': 186}}

In [29]:
from langchain_core.tools import tool


@tool
def multiply(a: int, b: int) -> int:
  """Get the product of two numbers"""
  return a * b



print(multiply.name)
print(multiply.description)
print(multiply.args)

multiply
Get the product of two numbers
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


In [3]:
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
prompt = PromptTemplate.from_template("你是一个{name}, 帮我起一个具有{country}特色的{sex}名字")
prompts = prompt.format(name="小明", country="中国", sex="男")
print(prompts)


chat_template = ChatPromptTemplate.from_messages(
  [
    ("system", "你是一个{name}, 帮我起一个具有{country}特色的{sex}名字"),
    ("user", "{input}"),
  ]
)

chat_templates = chat_template.format(name="小明", country="中国", sex="男", input="帮我起一个名字")
print(chat_templates)


你是一个小明, 帮我起一个具有中国特色的男名字
System: 你是一个小明, 帮我起一个具有中国特色的男名字
Human: 帮我起一个名字


In [7]:
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_core.prompts import MessagesPlaceholder


prompt_template = ChatPromptTemplate([
  ("system", "你是一个厉害的 AI 人工智能助手"),
  MessagesPlaceholder("msg")
])

result = prompt_template.invoke({"msg": [HumanMessage(content="Hi")] })
print(result)



sy = SystemMessage(
  content="你是一个大师",
  additional_kwargs={
    "大师名字": "陈瞎子"
  }
)

hu = HumanMessage(
  content="请问大师叫什么"
)

ai = AIMessage(
  content="我叫陈瞎子"
)

[sy, hu, ai]






messages=[SystemMessage(content='你是一个厉害的 AI 人工智能助手', additional_kwargs={}, response_metadata={}), HumanMessage(content='Hi', additional_kwargs={}, response_metadata={})]


[SystemMessage(content='你是一个大师', additional_kwargs={'大师名字': '陈瞎子'}, response_metadata={}),
 HumanMessage(content='请问大师叫什么', additional_kwargs={}, response_metadata={}),
 AIMessage(content='我叫陈瞎子', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]